# Brent · `status` (web: "Status")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**, so counts can differ slightly from the 2026-11-06
download.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · All `status` values

A `terms` aggregation on `status.raw` (exact values, case-sensitive) plus a `missing`
aggregation for records with no status. `size: 0` = no records, only the counts.

In [2]:
body = {
    "size": 0,
    "track_total_hits": True,
    "query": brent_filter(),
    "aggs": {
        "status": {"terms": {"field": "status.raw", "size": 100}},
        "no_status": {"missing": {"field": "status.raw"}},
    },
}
data = search(body)

total = data["hits"]["total"]["value"]
buckets = data["aggregations"]["status"]["buckets"]
status_counts = pd.DataFrame(
    [(b["key"], b["doc_count"]) for b in buckets] + [(None, data["aggregations"]["no_status"]["doc_count"])],
    columns=["status", "count"],
).sort_values("count", ascending=False, ignore_index=True)

print("Brent applications:", total, "| sum of counts:", status_counts["count"].sum())

Brent applications: 36952 | sum of counts: 36952


In [3]:
# From planninglondondatahub_public_technical_schemav2.1.xlsx, sheet "Field Values",
# index "applications", field "Status" (extracted 2026-11-09).
SCHEMA_STATUS = [
    "Allowed", "Appeal Received", "Application Under Consideration", "Approved",
    "Called in by Secretary of State", "Closed", "Comment Issued", "Declined to Determine",
    "Dismissed", "Appeal In Progress", "Insufficient Fee", "No Objection to Proposal (OBS only)",
    "Not Required", "Objection Raised to Proposal (OBS only)", "Opinion Issued",
    "Pre-application Advice Case Completed", "Refused", "Split Decision", "Unknown", "Withdrawn",
    "Application Invalid", "Application Received", "Quashed", "Commenced", "Completed", "Lapsed",
    "Superseded", "Revoked", "Pending Legacy Record", "Referred to Mayor",
]

status_counts["in_schema"] = status_counts["status"].isin(SCHEMA_STATUS)
status_counts["share_%"] = (100 * status_counts["count"] / total).round(2)
status_counts

,status,count,in_schema,share_%
0,Approved,20192,True,54.64
1,Refused,6949,True,18.81
2,Completed,3684,True,9.97
3,Lapsed,1810,True,4.90
4,Withdrawn,1446,True,3.91
5,Dismissed,719,True,1.95
6,Superseded,572,True,1.55
7,Application Received,459,True,1.24
8,NaN,418,False,1.13
9,Allowed,242,True,0.65


In [4]:
print("Seen in Brent but not in the schema:")
display(status_counts[~status_counts["in_schema"]])

unseen = sorted(set(SCHEMA_STATUS) - set(status_counts["status"].dropna()))
print(f"In the schema but never seen in Brent ({len(unseen)}):", unseen)

Seen in Brent but not in the schema:


,status,count,in_schema,share_%
8,NaN,418,False,1.13
13,lapsed,37,False,0.10
16,ZZZ,16,False,0.04
19,Appeal in Progress,2,False,0.01


In the schema but never seen in Brent (12): ['Called in by Secretary of State', 'Comment Issued', 'Declined to Determine', 'Insufficient Fee', 'Not Required', 'Opinion Issued', 'Pending Legacy Record', 'Pre-application Advice Case Completed', 'Quashed', 'Referred to Mayor', 'Revoked', 'Unknown']


### Findings · Step 1

Live API, 2026-11-09: **36,952** Brent applications, **22** distinct `status` values
(plus empty). The counts add up to the total.

- **Top 5 cover 92%:** `Approved` 54.6%, `Refused` 18.8%, `Completed` 10.0%,
  `Lapsed` 4.9%, `Withdrawn` 3.9%.
- **Not listed in the schema (4 values, 473 records):**

  | Value | Count | What it is |
  |---|---|---|
  | empty (null) | 418 | no status at all (Step 2) |
  | `lapsed` | 37 | lower-case variant of `Lapsed` |
  | `ZZZ` | 16 | not a status; a placeholder code (Step 2) |
  | `Appeal in Progress` | 2 | case variant of `Appeal In Progress` (1) |

- **Listed in the schema but never used in Brent (12):** `Called in by Secretary of State`,
  `Comment Issued`, `Declined to Determine`, `Insufficient Fee`, `Not Required`,
  `Opinion Issued`, `Pending Legacy Record`, `Pre-application Advice Case Completed`,
  `Quashed`, `Referred to Mayor`, `Revoked`, `Unknown`.
- `status` mixes **stages** (`Application Received`, `Commenced`, `Completed`),
  **council outcomes** (`Approved`, `Refused`, `Withdrawn`) and **appeal outcomes**
  (`Allowed`, `Dismissed`) in one field.
- Same as the 2026-11-06 download except `Application Received` 459 (was 460).

## Step 2 · Applications with `status` null or `ZZZ`

Fetch them all (a few hundred, well under the 10,000 window, so one request). Flat
fields only: the containers and polygons are excluded to keep it readable.

In [5]:
null_or_zzz_status = {
    "bool": {
        "should": [
            {"bool": {"must_not": [{"exists": {"field": "status.raw"}}]}},
            {"term": {"status.raw": "ZZZ"}},
        ],
        "minimum_should_match": 1,
    }
}
body = {
    "size": 2000,
    "track_total_hits": True,
    "query": {"bool": {"filter": [brent_filter(), null_or_zzz_status]}},
    "_source": {"excludes": ["application_details", "parking_details", "polygon", "wgs84_polygon"]},
    "sort": [{"id": "asc"}],
}
data = search(body)
hits = data["hits"]["hits"]
assert data["hits"]["total"]["value"] == len(hits), "more records than one page"

null_or_zzz = pd.DataFrame([h["_source"] for h in hits])
null_or_zzz["group"] = null_or_zzz["status"].fillna("null")
null_or_zzz["group"].value_counts()

group
null    418
ZZZ      16
Name: count, dtype: int64

In [6]:
# How full is each field, per group (% of records with a value).
def has_value(v: object) -> bool:
    # pandas turns missing values into NaN, so check for NaN as well as None / "" / [].
    if isinstance(v, (list, dict)):
        return len(v) > 0
    return pd.notna(v) and v != ""


def filled(series: pd.Series) -> float:
    return round(100 * series.map(has_value).mean(), 1)

fill = null_or_zzz.groupby("group").agg(filled).T
pd.set_option("display.max_rows", 100)
fill.sort_values(["null", "ZZZ"], ascending=False)

group,ZZZ,null
lpa_name,100.0,100.0
bo_system,100.0,100.0
application_type,100.0,100.0
last_updated_by,100.0,100.0
lpa_app_no,100.0,100.0
id,100.0,100.0
last_synced,100.0,100.0
last_updated,100.0,100.0
site_name,100.0,100.0
centroid,100.0,100.0


## `ZZZ` · Step 1 · All applications with status `ZZZ`

A `term` on `status.raw` (exact match), inside the Brent filter. The whole record is
kept, except the site polygons, because Step 2 needs `application_details`
(where `superseding_details` lives).

In [7]:
body = {
    "size": 100,
    "track_total_hits": True,
    "query": {"bool": {"filter": [brent_filter(), {"term": {"status.raw": "ZZZ"}}]}},
    "_source": {"excludes": ["polygon", "wgs84_polygon"]},
    "sort": [{"id": "asc"}],
}
data = search(body)
hits = data["hits"]["hits"]
assert data["hits"]["total"]["value"] == len(hits), "more records than one page"

zzz_apps = [h["_source"] for h in hits]
zzz = pd.DataFrame(zzz_apps)
print("Applications with status ZZZ:", len(zzz))
zzz[["id", "lpa_app_no", "valid_date", "decision", "decision_date", "appeal_decision",
     "appeal_decision_date", "site_name", "description"]]

Applications with status ZZZ: 16


,id,lpa_app_no,valid_date,decision,decision_date,appeal_decision,appeal_decision_date,site_name,description
0,Brent-18_0301,18/0301,16/02/2018,Refused,03/04/2018,ZZZ,08/08/2018,"1-4 Glengall House, Glengall Road, London, NW6...",Retrospective application for existing roof ex...
1,Brent-20_2457,20/2457,12/10/2020,Refused,07/12/2020,ZZZ,17/03/2021,"2 Oakington Manor Drive, Wembley, HA9 6LT","Demolition of single storey garage at front, p..."
2,Brent-20_2995,20/2995,22/09/2020,Refused,17/11/2020,ZZZ,28/05/2021,"69 Monks Park, Wembley, HA9 6JG",Proposed erection of dwelling house
3,Brent-20_2996,20/2996,29/09/2020,Refused,18/11/2020,ZZZ,03/06/2021,"69 Monks Park, Wembley, HA9 6JG",Demolition of garage and erection of a two sto...
4,Brent-20_3337,20/3337,15/10/2020,Refused,21/10/2020,ZZZ,08/02/2021,"76 Brondesbury Park, Brondesbury, London, NW2 5JU",Prior approval for a single storey rear extens...
5,Brent-20_3381,20/3381,20/10/2020,Refused,14/12/2020,ZZZ,28/06/2021,"52 Park Lane, Wembley, HA9 7RZ",Proposed demolition of outbuilding and excavat...
6,Brent-21_1554,21/1554,28/04/2021,Refused,23/06/2021,ZZZ,19/09/2021,"24 West Ella Road, London, NW10 9PU",Proposed roof extension with dormer window to ...
7,Brent-22_0506,22/0506,24/02/2022,Refused,09/05/2022,ZZZ,20/02/2023,"96 Village Way, London, NW10 0LL",Change of use of the property into a six-perso...
8,Brent-22_1501,22/1501,18/05/2022,Refused,13/07/2022,ZZZ,07/03/2023,"Land Adj to 5, Hawthorn Road, London",Erection of a one bedroom dwellinghouse with r...
9,Brent-22_2832,22/2832,11/08/2022,Refused,06/10/2022,ZZZ,08/06/2023,"Street Record, Ealing Road, Wembley",The removal of 2no. phone kiosks from the stre...


### Findings · `ZZZ` Step 1

**16** Brent applications have status `ZZZ` (live API, 2026-11-09).

- All 16 have `decision` = `Refused`, and `appeal_decision` is also `ZZZ`.
- All 16 have an `appeal_decision_date`, between 08/08/2018 and 11/09/2025.
- Valid dates run from 2018 to 2025, so this is not one old batch.
- They are mostly small schemes: extensions, single dwellings, changes of use,
  2 phone-kiosk removals.
- Two pairs share a site and were submitted close together:
  - `20/2995` and `20/2996`: 69 Monks Park, Wembley (valid 22/09/2020 and 29/09/2020);
  - `22/2832` and `22/2833`: Street Record, Ealing Road, Wembley (both valid 11/08/2022,
    same description: removal of 2 phone kiosks).

## `ZZZ` · Step 2 · Do they reference each other?

An application can point to another one in three places:
- `application_details.superseding_details[].lpa_app_no`: applications it replaces;
- `reference_no_of_permission_being_relied_on`: a permission it relies on;
- `description`: free text that may mention another reference, e.g. "21/3377".

Collect every reference from those three places, then check whether any of them is
the `lpa_app_no` of another `ZZZ` application.

In [8]:
import re

REF_PATTERN = re.compile(r"\b\d{2}/\d{4}\b")  # Brent's usual reference shape, e.g. 21/3377

rows = []
for app in zzz_apps:
    details = app.get("application_details") or {}
    for item in details.get("superseding_details") or []:
        rows.append((app["id"], "superseding_details", item.get("lpa_app_no")))
    if app.get("reference_no_of_permission_being_relied_on"):
        rows.append((app["id"], "relied_on", app["reference_no_of_permission_being_relied_on"]))
    for ref in REF_PATTERN.findall(app.get("description") or ""):
        if ref != app["lpa_app_no"]:
            rows.append((app["id"], "description", ref))

refs = pd.DataFrame(rows, columns=["from_id", "where", "to_lpa_app_no"])
zzz_refs = set(zzz["lpa_app_no"])
refs["to_is_zzz"] = refs["to_lpa_app_no"].isin(zzz_refs)

print("References found:", len(refs), "| pointing to another ZZZ application:", int(refs["to_is_zzz"].sum()))
refs

References found: 0 | pointing to another ZZZ application: 0


,from_id,where,to_lpa_app_no,to_is_zzz


### Findings · `ZZZ` Step 2

**No references at all**: none of the 16 has `superseding_details`, a
`reference_no_of_permission_being_relied_on`, or another reference number in its
`description`. So they don't reference each other (or anything else) in the data.

The only connection between some of them is the **same site** (the two pairs in
Step 1), which the data doesn't record as a link.

## `ZZZ` · Step 3 · Appeal start and appeal decision dates

The two appeal dates for all 16 `ZZZ` applications (from Step 1, no new request),
then the one application that has an appeal start date.

In [9]:
zzz[["id", "appeal_start_date", "appeal_decision_date"]]

,id,appeal_start_date,appeal_decision_date
0,Brent-18_0301,NaN,08/08/2018
1,Brent-20_2457,NaN,17/03/2021
2,Brent-20_2995,NaN,28/05/2021
3,Brent-20_2996,NaN,03/06/2021
4,Brent-20_3337,NaN,08/02/2021
5,Brent-20_3381,NaN,28/06/2021
6,Brent-21_1554,NaN,19/09/2021
7,Brent-22_0506,13/07/2022,20/02/2023
8,Brent-22_1501,NaN,07/03/2023
9,Brent-22_2832,NaN,08/06/2023


In [10]:
with_start = zzz[zzz["appeal_start_date"].notna()]
print("ZZZ applications with an appeal start date:", len(with_start))
with_start.drop(columns=["application_details", "parking_details"]).T

ZZZ applications with an appeal start date: 1


,7
lpa_name,Brent
bo_system,CSV Data Import
title_number,[TBC111]
decision_process,Delegated
secondary_street_name,None
borough,Brent
application_type_full,Full planning permission
decision_agency,None
decision_target_date,09/05/2022
last_updated_by,CSV Data Import


### Findings · `ZZZ` Step 3

- **Appeal start date:** only **1 of 16** has one. The other 15 are empty.
- **Appeal decision date:** **all 16** have one, from 08/08/2018 to 11/09/2025.
- **The one with a start date: `Brent-22_0506`** (96 Village Way, NW10 0LL, Welsh Harp
  ward), "Change of use of the property into a six-person HMO (Use Class: C4)":

  | Date | Field |
  |---|---|
  | 24/02/2022 | `valid_date` |
  | 09/05/2022 | `decision_date` (`Refused`, delegated) |
  | 13/07/2022 | `appeal_start_date` |
  | 20/02/2023 | `appeal_decision_date` (`appeal_decision` = `ZZZ`) |

  The dates are in a sensible order, and the appeal took about 7 months. Apart from
  the `ZZZ` codes, this record looks like an ordinary refused-then-appealed application.
- Oddities in the same record: `title_number` is `["TBC111"]` (looks like a "to be
  confirmed" placeholder), and the building-work dates under a previous permission
  are both 04/02/2016.

## Empty status · Step 1 · What did the council decide?

The `decision` values of the applications with no `status` (from Step 2's
`null_or_zzz`, no new request). `dropna=False` so that empty decisions are counted too.

In [11]:
empty_status = null_or_zzz[null_or_zzz["group"] == "null"]
print("Applications with empty status:", len(empty_status))
empty_status["decision"].value_counts(dropna=False)

Applications with empty status: 418


decision
NaN    418
Name: count, dtype: int64

### Findings · Empty status Step 1

**No decision at all.** All 418 applications with an empty `status` also have an empty
`decision`. The data doesn't say what the council decided for any of them, so neither
field can tell whether they were approved, refused or are still pending.

## Empty status · Step 2 · Duplicate check

Do the 407 empty records (no status, no description) have a real copy among the
records with content? Compared with all Brent applications from the live API, a few
fields only, paged with `search_after` (sorted by `id`) because there are more than
10,000.

Matches, strongest first:
- **A. same `pp_id`**: the same Planning Portal submission, registered twice;
- **B. same site, same reference year, same `application_type`, reference number within
  50**: probably the same application;
- C. same site and year, otherwise different: other applications at that address;
- D. same site, another year;
- E. no match.

The reference number alone is **not** used: Brent numbers its `/TELN` series
separately, so `20/0070/TELN` and `20/0070` are different applications.

In [12]:
FIELDS = ["id", "lpa_app_no", "pp_id", "site_name", "application_type", "status", "description"]

rows, after = [], None
while True:
    body = {
        "size": 5000,
        "query": brent_filter(),
        "_source": FIELDS,
        "sort": [{"id": "asc"}],
    }
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

brent = pd.DataFrame(rows).reindex(columns=FIELDS)
print("Brent applications fetched:", len(brent))

Brent applications fetched: 36952


In [13]:
import re


def norm_site(s: object) -> str:
    # Compare addresses without case, spaces or punctuation.
    return re.sub(r"[^a-z0-9]", "", str(s).lower()) if pd.notna(s) else ""


def ref_number(ref: str) -> int | None:
    m = re.match(r"\d{2}/(\d+)", ref)
    return int(m.group(1)) if m else None


brent["site_key"] = brent["site_name"].map(norm_site)
brent["ref_year"] = brent["lpa_app_no"].str[:2]
brent["ref_no"] = brent["lpa_app_no"].map(ref_number)

empty = brent[brent["status"].isna() & brent["description"].isna()]
full = brent[brent["description"].notna()]
print("Empty records:", len(empty), "| records with content:", len(full))

pp_full = full[full["pp_id"].notna()].groupby("pp_id")["id"].first()
site_full = full.groupby("site_key")


def classify(e: pd.Series) -> tuple[str, str | None]:
    if pd.notna(e["pp_id"]) and e["pp_id"] in pp_full.index:
        return "A same pp_id", pp_full[e["pp_id"]]
    if e["site_key"] not in site_full.groups:
        return "E no match", None
    same_site = site_full.get_group(e["site_key"])
    same_year = same_site[same_site["ref_year"] == e["ref_year"]]
    if same_year.empty:
        return "D same site, other year", None
    close = same_year[
        (same_year["application_type"] == e["application_type"])
        & ((same_year["ref_no"] - e["ref_no"]).abs() <= 50)
    ]
    if not close.empty:
        return "B same site+year+type, ref within 50", close["id"].iloc[0]
    return "C same site+year, otherwise different", None


matches = empty.apply(classify, axis=1, result_type="expand")
empty = empty.assign(match=matches[0], duplicate_of=matches[1])
empty["match"].value_counts().sort_index()

Empty records: 407 | records with content: 36544


match
A same pp_id                              15
B same site+year+type, ref within 50       9
C same site+year, otherwise different    165
D same site, other year                   88
E no match                               130
Name: count, dtype: int64

In [14]:
# The duplicates (A and B) next to the record they duplicate.
dups = empty[empty["match"].str[0].isin(["A", "B"])]
dups = dups.merge(
    full[["id", "status", "site_name"]].rename(columns=lambda c: f"full_{c}"),
    left_on="duplicate_of", right_on="full_id",
)
dups[["match", "id", "site_name", "full_id", "full_status", "full_site_name"]]

,match,id,site_name,full_id,full_status,full_site_name
0,"B same site+year+type, ref within 50",Brent-21_0094,"93 Melrose Avenue, London, NW2 4LR",Brent-21_0092,Refused,"93 Melrose Avenue, London, NW2 4LR"
1,A same pp_id,Brent-21_0224,"18 Highcroft Avenue, Wembley, HA0 1TG",Brent-21_0574,Allowed,"18 Highcroft Avenue, Wembley, HA0 1TG"
2,"B same site+year+type, ref within 50",Brent-21_0402,"399 Edgware Road, London, NW9 0FH",Brent-21_0435,Refused,"399 Edgware Road, London, NW9 0FH"
3,A same pp_id,Brent-21_0763,"1 Bridgehill Close, Wembley, HA0 1EP",Brent-21_0764,Approved,"1-7 Bridgehill Close, Wembley, HA0 1EP"
4,"B same site+year+type, ref within 50",Brent-21_1014,"11 Hill View Gardens, London, NW9 0TE",Brent-21_0998,Refused,"11 Hill View Gardens, London, NW9 0TE"
5,"B same site+year+type, ref within 50",Brent-21_1330,"14 Alverstone Road, London, NW2 5JT",Brent-21_1329,Refused,"14 Alverstone Road, London, NW2 5JT"
6,A same pp_id,Brent-21_1348,"21 Neeld Crescent, Wembley, HA9 6LP",Brent-21_1315,Refused,"21 Neeld Crescent, Wembley, HA9 6LP"
7,A same pp_id,Brent-21_2265,"164 High Road, London, NW10 2PB",Brent-21_2261,Completed,"164 High Road, London, NW10 2PB"
8,A same pp_id,Brent-21_2427,"Ashford House, Ashford Road, London, NW2 6TL",Brent-21_2430,Withdrawn,"1-6 Ashford House, Ashford Road, London, NW2 6TL"
9,"B same site+year+type, ref within 50",Brent-21_2499,"247A Neasden Lane, London, NW10 1QG",Brent-21_2530,Refused,"247A Neasden Lane, London, NW10 1QG"


### Findings · Empty status Step 2

Live API, 2026-11-09: 36,952 Brent applications, 407 empty records, 36,544 with content.

| Match | Empty records | Meaning |
|---|---|---|
| **A. same `pp_id`** | **15** | the same Planning Portal submission, registered twice → **duplicate** |
| **B. same site + year + type, ref within 50** | **9** | very likely the same application → **duplicate** |
| C. same site + year, otherwise different | 165 | other applications at the address, not duplicates |
| D. same site, other year | 88 | not duplicates |
| E. no match | 130 | nothing else at the address |

- **24 duplicates (A + B)**, all with `21/` or `22/` references. Each one points to a
  full record that has a real status (Approved, Refused, Allowed, Dismissed, Lapsed,
  Withdrawn, Completed), e.g. `Brent-21_0224` → `Brent-21_0574` (18 Highcroft Avenue,
  Allowed).
- Two full records have **two** empty copies each: `Brent-21_3883` (36 Beaconsfield
  Road) and `Brent-22_0422` (224B Walm Lane).
- A few `pp_id` matches have a slightly different address text (`1 Bridgehill Close`
  vs `1-7 Bridgehill Close`), but the same submission number.
- **The other 383 have no copy.** They are the only trace of whatever they were.

**Decision (2026-11-09):**
- the 24 duplicates are **removed** (the full record they point to is kept);
- the remaining empty-status records get the clean status **`Unknown`** (a value
  already listed in the schema).

## Type check · Do the values match the schema type?

The schema gives `status` the type `string`. Earlier counts used an Elasticsearch
aggregation (or `value_counts`), which can't show whether a value came as a JSON string or a
JSON number. Count the Python type of every value: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [15]:
values, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "status"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    values += [h["_source"] for h in page]
    after = page[-1]["sort"]

typed = pd.DataFrame(values).reindex(columns=["id", "status"])
types = typed["status"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
print("Brent applications:", len(typed))
display(types.value_counts().to_frame("count"))

not_str = typed[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["status"].value_counts().head(30).to_frame("count")

Brent applications: 36952


,count
status,
str,36534
NoneType,418


values that are not strings: 0


,count
status,


### Findings · Type check

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 36,534 |
| null | 418 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says. No type mismatch.